# Semana 2 — Transformación del Dataset para Machine Learning

**Objetivo:** Transformar el dataset original (formato largo, 2000–2022) a un formato utilizable para regresión.

**Problema a resolver:** Estimar el porcentaje de uso de Internet por país, año y grupo etario.

**Equipo de trabajo:**
- **Persona 1:** Diseño del dataset transformado
- **Persona 2:** Implementación de transformaciones
- **Persona 3:** Validación de calidad y valores faltantes
- **Persona 4:** Integración y entrega final

---

**Archivo principal:** `02_semana2_transformacion.ipynb`  
**Dataset transformado:** `datos_transformados_semana2.csv`

## PERSONA 1: Diseño del Dataset Transformado

### 1. Análisis del Formato Original

El dataset original está en formato **largo**, donde cada fila representa una observación de país, año y grupo etario. Esta estructura es ideal para exploración visual pero **no es suficiente para regresión** por las siguientes razones:

- **Carencia de variables temporales explícitas:** El año es una dimensión de la tabla, no una variable numérica que capture tendencia.
- **Sin estructura tabular directa:** Para regresión supervisa necesitamos variables independientes en columnas y variable objetivo aislada.
- **Columnas redundantes:** `indicator`, `unit`, `notes_ids`, `source_id` tienen valores fijos o vacíos sin aportar señal predictiva.
- **Panel desbalanceado:** Requiere limpieza (valores faltantes, período confiable desde 2016).

**Recomendación de Semana 1:** Pivotaje a ancho + codificación categórica + variables temporales.

### 2. Clasificación de Columnas del Dataset Original

| Columna | Tipo | Decisión | Justificación |
|---------|------|----------|---------------|
| `indicator` | string (fijo) | **ELIMINAR** | Valor constante; sin variabilidad para ML |
| `País__ESTANDAR` | string | **CONSERVAR** | Dimensión clave del problema; diferencia importante entre países |
| `Grupos etarios Uso Internet` | string | **CONSERVAR** (sin "Total") | Dimensión clave; modelar por grupo etario específico |
| `Años__ESTANDAR` | numeric | **CONSERVAR + TRANSFORMAR** | Base temporal; crear `years_since_2016` para capturar tendencia |
| `value` | numeric (0–100) | **CONSERVAR** | Variable objetivo; porcentaje de usuarios de Internet |
| `unit` | string (fijo) | **ELIMINAR** | Valor constante; sin valor predictivo |
| `notes_ids` | string (vacío) | **ELIMINAR** | Mayormente faltante; sin valor para modelo |
| `source_id` | numeric (fijo) | **ELIMINAR** | Identificador de fuente; no añade información analítica |

### 3. Propuesta de Estructura Final del Dataset

#### Unidad de análisis
**Cada fila = una observación de país, año y grupo etario con su tasa de uso de Internet.**

**Período:** 2016–2022 (confiabilidad post-2016, Semana 1)  
**Filtros:** Sin grupo "Total"; solo grupos etarios específicos

#### Columnas del CSV Transformado

| Columna | Tipo | Rango/Valores | Descripción |
|---------|------|---------------|-------------|
| `pais` | string | 13 países únicos | Nombre estandarizado del país (de `País__ESTANDAR`) |
| `año` | int | 2016–2022 | Año del registro (de `Años__ESTANDAR`, filtrado) |
| `years_since_2016` | int | 0–6 | Años transcurridos desde 2016 (variable temporal derivada) |
| `grupo_etario` | string | 5 grupos | Grupo de edad (de `Grupos etarios Uso Internet`, sin "Total") |
| `porcentaje_internet` | float | 0.0–100.0 | Porcentaje de usuarios de Internet (**variable objetivo**) |

#### Tamaño esperado
- Combinaciones posibles: 13 países × 7 años × 5 grupos = **455 filas** (sin valores faltantes)
- Estructura resultante: **455 × 5** (455 observaciones, 5 columnas)
- Formato: tabular directo, listo para regresión supervisada

### 4. Justificación del Diseño Propuesto

#### ¿Por qué esta estructura es mejor que el original?

1. **Tabular y limpia**
   - Cada fila es independiente (observación completa)
   - Todas las variables en columnas, lista para modelos de ML
   - Sin ruido de columnas redundantes

2. **Captura de tendencia temporal**
   - `years_since_2016` es una variable numérica derivada que permite al modelo aprender aceleración/desaceleración
   - Facilita la captura de dinámicas entre 2016 y 2022

3. **Alineación con el problema**
   - Problema: "Regresión para estimar porcentaje por país, año y grupo"
   - Diseño: cada fila tiene país, año (+ tendencia), grupo y objetivo
   - Estructura 1-a-1 con los requisitos del problema

4. **Período confiable**
   - Solo 2016–2022: datos post-2016 tienen mayor cobertura y consistencia (Semana 1)
   - Reduce ruido de períodos con baja confiabilidad

5. **Sin variables redundantes**
   - Eliminadas columnas con valores fijos (`indicator`, `unit`, `source_id`)
   - Eliminada columna mayormente vacía (`notes_ids`)
   - Reducción de dimensionalidad → mejor generalización del modelo

#### Decisiones clave registradas

- ✅ **Unidad de análisis:** país-año-grupo  
- ✅ **Período:** 2016–2022  
- ✅ **Filtro:** Sin grupo "Total"  
- ✅ **Variables derivadas:** Solo `years_since_2016` (Persona 2 añade dummies/retardos si necesita)  
- ✅ **Columnas a eliminar:** `indicator`, `unit`, `notes_ids`, `source_id`

---

## PERSONA 2: Transformación del Dataset

### Objetivo de implementación
Aplicar las transformaciones definidas en el diseño de Persona 1 para construir una primera versión del dataset listo para machine learning.

### Flujo de transformación
1. Cargar `data/datos.csv` con separador correcto.
2. Filtrar período 2016-2022 y excluir grupo `Total`.
3. Renombrar columnas para claridad y consistencia.
4. Crear variable derivada `years_since_2016`.
5. Validar estructura resultante (duplicados, rango y combinaciones).
6. Exportar primera versión del CSV transformado.

### Resultado esperado
Un dataset tabular limpio, con menos columnas que el original y estructura utilizable para modelado supervisado.

### 5. Implementación paso a paso (Persona 2)

#### 5.1 Carga de datos
Se carga el dataset original y se verifica su estructura inicial para documentar el punto de partida.

In [1]:
import pandas as pd

ruta_origen = "../data/datos.csv"
df_raw = pd.read_csv(ruta_origen, sep=";", encoding="latin1")

print(f"Filas originales: {len(df_raw)}")
print(f"Columnas originales: {df_raw.shape[1]}")
df_raw.head()

Filas originales: 870
Columnas originales: 8


,indicator,País__ESTANDAR,Grupos etarios Uso Internet,Años__ESTANDAR,value,unit,notes_ids,source_id
0,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2016,76,Porcentaje sobre el total de personas en cada ...,NaN,9353
1,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2017,76,Porcentaje sobre el total de personas en cada ...,NaN,9353
2,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2018,79,Porcentaje sobre el total de personas en cada ...,NaN,9353
3,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2019,79,Porcentaje sobre el total de personas en cada ...,NaN,9353
4,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2020,88,Porcentaje sobre el total de personas en cada ...,NaN,9353


#### 5.2 Filtrado, selección y renombrado
Se filtra el período 2016-2022, se excluye el grupo `Total`, se conservan columnas útiles y se renombran para consistencia.

In [2]:
columnas_utiles = [
    "País__ESTANDAR",
    "Grupos etarios Uso Internet",
    "Años__ESTANDAR",
    "value",
]

df = df_raw[columnas_utiles].rename(
    columns={
        "País__ESTANDAR": "pais",
        "Grupos etarios Uso Internet": "grupo_etario",
        "Años__ESTANDAR": "anio",
        "value": "porcentaje_internet",
    }
)

df["anio"] = pd.to_numeric(df["anio"], errors="coerce")
df["porcentaje_internet"] = pd.to_numeric(df["porcentaje_internet"], errors="coerce")

df = df[(df["anio"] >= 2016) & (df["anio"] <= 2022)]
df = df[df["grupo_etario"].str.strip().str.lower() != "total"].copy()

print(f"Filas luego de filtrar: {len(df)}")
print(f"Columnas luego de filtrar: {df.shape[1]}")
df.head()

Filas luego de filtrar: 340
Columnas luego de filtrar: 4


,pais,grupo_etario,anio,porcentaje_internet
0,Argentina,edad de medicion a 17 años,2016,76
1,Argentina,edad de medicion a 17 años,2017,76
2,Argentina,edad de medicion a 17 años,2018,79
3,Argentina,edad de medicion a 17 años,2019,79
4,Argentina,edad de medicion a 17 años,2020,88


#### 5.3 Reestructuración para machine learning
Se crea una variable temporal y se codifica el grupo etario en variables dummies para dejar una matriz más útil para modelos.

In [3]:
df["years_since_2016"] = df["anio"] - 2016

# Estandariza el texto para mejorar consistencia de categorías
df["grupo_etario"] = (
    df["grupo_etario"]
    .str.replace("á", "a", regex=False)
    .str.replace("é", "e", regex=False)
    .str.replace("í", "i", regex=False)
    .str.replace("ó", "o", regex=False)
    .str.replace("ú", "u", regex=False)
    .str.replace("ñ", "n", regex=False)
    .str.strip()
    .str.lower()
)

df_ml = pd.get_dummies(
    df,
    columns=["grupo_etario"],
    prefix="grupo",
    dtype=int,
)

columnas_base = ["pais", "anio", "years_since_2016", "porcentaje_internet"]
columnas_dummies = sorted([c for c in df_ml.columns if c.startswith("grupo_")])
df_ml = df_ml[columnas_base + columnas_dummies].sort_values(["pais", "anio"]).reset_index(drop=True)

print(f"Dataset transformado (ML): {df_ml.shape[0]} filas x {df_ml.shape[1]} columnas")
df_ml.head()

Dataset transformado (ML): 340 filas x 9 columnas


,pais,anio,years_since_2016,porcentaje_internet,grupo_18 a 25 anos de edad,grupo_26 a 50 anos de edad,grupo_51 a 65 anos,grupo_66 anos en adelante,grupo_edad de medicion a 17 anos
0,Argentina,2016,0,76,0,0,0,0,1
1,Argentina,2016,0,86,1,0,0,0,0
2,Argentina,2016,0,82,0,1,0,0,0
3,Argentina,2016,0,61,0,0,1,0,0
4,Argentina,2016,0,29,0,0,0,1,0


#### 5.4 Validación rápida de la transformación
Se verifica que no queden duplicados por observación y se calcula cobertura esperada por combinaciones país-año-grupo.

In [4]:
duplicados = df.duplicated(subset=["pais", "anio", "grupo_etario"]).sum()
faltantes = df_ml.isna().sum().sum()

n_paises = df["pais"].nunique()
n_anios = df["anio"].nunique()
n_grupos = df["grupo_etario"].nunique()
esperado = n_paises * n_anios * n_grupos

print(f"Duplicados (pais-anio-grupo): {duplicados}")
print(f"Valores faltantes en dataset ML: {faltantes}")
print(f"Cobertura observada: {len(df)} / {esperado} combinaciones esperadas")
print(f"Paises: {n_paises}, Años: {n_anios}, Grupos etarios: {n_grupos}")

Duplicados (pais-anio-grupo): 0
Valores faltantes en dataset ML: 0
Cobertura observada: 340 / 455 combinaciones esperadas
Paises: 13, Años: 7, Grupos etarios: 5


#### 5.5 Exportación de primera versión
Se exporta la primera versión del dataset transformado en formato CSV para revisión del equipo.

In [5]:
ruta_salida = "../outputs/datos_transformados_semana2.csv"
df_ml.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"CSV exportado en: {ruta_salida}")
print("Columnas finales:")
print(df_ml.columns.tolist())

CSV exportado en: ../outputs/datos_transformados_semana2.csv
Columnas finales:
['pais', 'anio', 'years_since_2016', 'porcentaje_internet', 'grupo_18 a 25 anos de edad', 'grupo_26 a 50 anos de edad', 'grupo_51 a 65 anos', 'grupo_66 anos en adelante', 'grupo_edad de medicion a 17 anos']


---

## PERSONA 3: Validación de Calidad

*(Pendiente de implementación por Persona 3)*

---

## PERSONA 4: Integración y Entrega Final

*(Pendiente de implementación por Persona 4)*